# Isotope graph generator

### Input

The following notebook takes 2 files as input:

1. The [isotope curves](https://docs.google.com/spreadsheets/d/1P-NDhxRnLU8Tkg4dlXvef7DCYKE2qrSx/edit?usp=drive_link&ouid=102758595970305145284&rtpof=true&sd=true) for d13C and d18O per marble type, extracted from the literature.
2. [Isotope values for marble samples](https://docs.google.com/spreadsheets/d/1N26mnpoRzENkesBL3uAAFfRp2uHCR0Dq/edit?usp=drive_link&ouid=102758595970305145284&rtpof=true&sd=true) from the project, arranged by ISIC code.

Both of these files are read first by mounting Google Drive, and then by filename.

*Note*: In the future, file (2) above will instead be pulled from two columns (d13C and d18O) from [the larger marble sample spreadsheet](https://docs.google.com/spreadsheets/d/1c3LYccRJfIGlzZRTrQSkYizLQ0eCW3F2l3lTIjilvsM/edit?usp=drive_link), but these are not yet inputted.


### Output

The notebook then produces two forms of output:

1. A graphical representation of the isotope curves (1), portrayed as smoothed polygons, over which the marble sample values (2) are displayed as points. This graph is displayed below as well as saved in the Crossreads B Drive folder as:

    * [isotope_graph.png](https://drive.google.com/file/d/1-0nj3_YqpNqVg1j_g0b56wqLKP0JtuSF/view?usp=drive_link) (an image)
    * [isotope_graph.pdf](https://drive.google.com/file/d/1-0aTeIeJHVrnTBLcrGN29rR9g4eolsd8/view?usp=drive_link) (a pdf document)
    * [isotope_graph.html](https://drive.google.com/file/d/137o7AhyZ_LqfdfHxBDgJKi5SET8lzeiz/view?usp=drive_link) (an interactive graph HTML file; you'll need to download it and double-click on your own machine for it to work)

2. A tabular representation of which marble type polygons a given marble sample is contained within, sorted by the sample's distance from the marble type polygon's centroid. This table is shown below as well as saved to:

    * [isotope_intersections.xlsx](https://docs.google.com/spreadsheets/d/1-Dw9OFwnKKdnrWv1tWI_44h-RILLI0fZ/edit?usp=drive_link&ouid=102758595970305145284&rtpof=true&sd=true) (an excel file / Google sheets spreadsheet)

Please contact [Ryan Heuser](mailto:ryan.heuser@kcl.ac.uk) with any issues or recommendations for improvement.

In [1]:
import sys; sys.path.append('..')
from crossreads_petrography.isotopes import IsotopeConverter

In [2]:
isotoper = IsotopeConverter()

Initializing IsotopeConverter @ 2024-07-16 19:44:04,920
Authenticating and accessing Google Spreadsheet @ 2024-07-16 19:44:04,921


In [3]:
isotoper.read_isotope_data()

Reading isotope data from Google Sheets @ 2024-07-16 19:44:06,421


APIError: APIError: [400]: This operation is not supported for this document

In [1]:
# @title Setup code and get isotope valus from Crossreads spreadsheet
!pip install -q plotly shapely
import pandas as pd
import numpy as np
import os
from shapely.geometry import Point, Polygon
import plotly.graph_objects as go
from scipy.interpolate import splprep, splev
import plotly.io as pio
import gspread
from google.auth import default
from google.colab import auth, drive



# Get from google drive folder
drive.mount('/content/drive')
folder='/content/drive/MyDrive/Crossreads B D1/'
folder_curves=os.path.join(folder,'Isotope input data')
fn_samples='isotopes-1.xlsx'  # instead of pulling from here, can now pull from v2 spreadsheet, columns
# y = isotopes delta13C,
# x = isotopes delta18O

# load big spreadsheet
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1Yqxm6pwNAmz8GJDbcqOphNG9WB9xLif8u7sMplVIkqs/edit')
worksheet = spreadsheet.get_worksheet(0)

# Get all data from the worksheet
rows = worksheet.get_all_values()
df_big = pd.DataFrame.from_records(rows)

# Set the first row as the header if needed
df_big.columns = df_big.iloc[0]
df_big = df_big.drop(0)
xcol='isotopes delta13C'
ycol='isotopes delta18O'

df_points = df_big.set_index(df_big.columns[0])[[xcol,ycol]]
df_points['Sample'] = df_points.index
df_points = df_points[~df_points.Sample.str.contains(' ')]
df_points['y'] = df_points[xcol]
df_points['x'] = df_points[ycol]
df_points=df_points.reset_index()[['Sample','x','y']]
df_points=df_points.query('Sample!="" & x!="" & y!=""')
df_points

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 6.9 MB/s eta 0:00:00
Mounted at /content/drive


,Sample,x,y
1,ISic000104,-2.29,1.92
3,ISic000097,-1.49,3.71
4,ISic000004,-2.69,2.39
5,ISic000034,-0.71,2.56
6,ISic000036,-1.35,2.72
...,...,...,...
92,EXMFT109,-2.01,2.26
93,EXMFT112,-2.18,2.26
94,EXMFT134,-2.87,2.54
95,ICT005,-1.78,1.96


In [2]:
# @title Reshape curve data from original format
df_curves=pd.concat(
    pd.read_excel(os.path.join(folder_curves,fn_curves))
    for fn_curves in os.listdir(folder_curves)
    if fn_curves.endswith('.xlsx')
)
# df_points = pd.read_excel(os.path.join(folder, fn_samples))


o=[]
types={x.split('_')[0] for x in df_curves.columns}
for i,row in df_curves.iterrows():
    for typename in types:
        d={}
        d['marble_type'] = typename
        for coord in ['x','y']:
            colname=f'{typename}_{coord}'
            d[coord]=row[colname]
        o.append(d)
df_curves_reshaped = pd.DataFrame(o).dropna()
df_curves_reshaped

,marble_type,x,y
0,Thasos-1 (2),-3.445693,3.894349
1,Hymettus,-4.422843,2.293689
2,Aphrodisias,-6.529338,1.953317
3,Proconnesos-2,-9.126092,2.862408
4,Naxos,-13.932584,2.653563
...,...,...,...
4066,EphesosWhites1,-9.307172,3.005405
4086,EphesosWhites1,-9.144790,3.135135
4106,EphesosWhites1,-8.982409,3.254054
4126,EphesosWhites1,-8.779432,3.362162


In [3]:
# @title Plot curves

def plot_curves(df):

    df = df_curves_reshaped

    # Create a plotly figure
    fig = go.Figure()

    # Group by marble_type and plot each group using the original points
    for marble_type, group in df.groupby('marble_type'):
        # Use original points order, but ensure to close the polygon by appending the first point
        x_closed = np.append(group['x'].values, group['x'].values[0])
        y_closed = np.append(group['y'].values, group['y'].values[0])
        fig.add_trace(go.Scatter(
            x=x_closed,
            y=y_closed,
            fill='toself',
            name=marble_type,
            mode='lines'
        ))


    fig.add_trace(go.Scatter(
        x=df_points['x'],
        y=df_points['y'],
        mode='markers+text',
        text=df_points['Sample'],
        textposition='top center',
        marker=dict(size=10, color='red', symbol='circle'),
        name='Samples'
    ))

    # Update layout
    fig.update_layout(
        title='Polygons for each marble type + points for marble samples',
        xaxis_title='d18O',
        yaxis_title='d13C',
        showlegend=True,
        height=800,
        width=1000
    )

    # Show the figure
    pio.write_html(fig, os.path.join(folder,'isotope_graph.html'))
    fig.write_image(os.path.join(folder,'isotope_graph.png'))
    fig.write_image(os.path.join(folder,'isotope_graph.pdf'))
    fig.show()

plot_curves(df_curves_reshaped)

In [5]:
# @title Determine polygon intersections

def determine_polygon_intersections(df_curves, df_points):
    # Create polygons from the original points and check which points are inside
    polygons = {}
    for marble_type, group in df_curves.groupby('marble_type'):
        # Using original points order to create polygons
        polygon = Polygon(zip(group['x'].values, group['y'].values))
        polygons[marble_type] = polygon

    # Initialize DataFrame to store results
    samples_list = df_points['Sample'].unique()
    marble_types = df_curves['marble_type'].unique()
    results_df = pd.DataFrame(index=samples_list, columns=marble_types)
    results_df = results_df.fillna('')  # Fill empty cells with empty string

    # Determine which polygon(s) each sample point is in
    for idx, row in df_points.iterrows():
        point = Point(row['x'], row['y'])
        intersected = False
        for marble_type, poly in polygons.items():
            if poly.contains(point):
                results_df.at[row['Sample'], marble_type] = '✔️'  # Set emoji if intersection is found
                intersected = True
        if not intersected:
            results_df.loc[row['Sample']] = results_df.loc[row['Sample']].replace('', '✖️')  # Optional: mark no intersection with an 'X'

    # Export to Excel
    if not results_df.empty:
        results_df = results_df[sorted(results_df.columns)]
        results_df.to_excel(os.path.join(folder, 'isotope_intersections.xlsx'))

    return results_df

# Assuming df_curves_reshaped and df_points are already defined and loaded
determine_polygon_intersections(df_curves_reshaped, df_points)

,Aphrodisias,CapDeGardeScritto,Carrara,Docimium,EphesosBigio,EphesosScritto,EphesosWhites1,EphesosWhites2,FilfilaScritto,Göktepe,Hymettus,Naxos,Paros-1,Paros-2 (3),Paros-4,Pentelikon,Proconnesos-1,Proconnesos-2,Thasos-1 (2),Thasos-3
ISic000104,✔️,,✔️,,,,,,✔️,✔️,✔️,,✔️,✔️,,,✔️,,✔️,
ISic000097,,,,,,,✔️,,,,,,,,,,✔️,,✔️,✔️
ISic000004,✔️,,✔️,✔️,,✔️,,,✔️,✔️,✔️,✔️,✔️,✔️,,,✔️,,✔️,
ISic000034,,,✔️,,,,,,,,✔️,,,✔️,,,✔️,,✔️,
ISic000036,,,✔️,,,,,,✔️,,✔️,,,✔️,,,✔️,,✔️,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EXMFT109,,,✔️,,,,,,✔️,✔️,✔️,,✔️,✔️,,,✔️,,✔️,
EXMFT112,,,✔️,✔️,,,,,✔️,✔️,✔️,,✔️,✔️,,,✔️,,✔️,
EXMFT134,✔️,,✔️,✔️,,✔️,,,✔️,✔️,✔️,✔️,✔️,✔️,,,✔️,,✔️,✔️
ICT005,,,✔️,,,,,,✔️,,✔️,,✔️,✔️,,,✔️,,✔️,
